# Comparing Classifiers for Bank Marketing Campaigns

## Business Understanding
The goal of this project is to help a bank improve its telephone marketing campaign by predicting whether a customer will subscribe to a term deposit. If the bank can identify likely subscribers in advance, it can target customers more efficiently, reduce campaign costs, and improve marketing success.

## Problem Statement
Compare the performance of K-Nearest Neighbors, Logistic Regression, Decision Trees, and Support Vector Machines on the bank marketing dataset and determine which model is most appropriate for the business problem.

## Evaluation Metric
Because the target classes are imbalanced, this analysis focuses on:
- Recall
- F1-score
- ROC-AUC

Recall is especially important because failing to identify a customer who would subscribe means losing a potential conversion.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)


## Data Loading

Place the dataset in the `data/` folder. Update the filename below if your class uses a different version such as `bank-full.csv`.


In [ ]:
# Update the file path if needed
df = pd.read_csv("data/bank-additional-full.csv", sep=";")
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())


## Initial Observations
- The target variable is `y`, which indicates whether the client subscribed to a term deposit.
- The data includes both categorical and numerical variables.
- Some categorical variables may include `"unknown"` values, which should be treated carefully.
- The dataset is used for binary classification.


In [ ]:
# Replace 'unknown' with NaN in object columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].replace("unknown", np.nan)

df.isnull().sum().sort_values(ascending=False).head(15)


## Feature Selection Note
The UCI documentation notes that `duration` should be excluded for realistic predictive modeling because it is only known after a call ends. Including it would create information leakage.


In [ ]:
if "duration" in df.columns:
    df = df.drop(columns=["duration"])

df.shape


## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="y")
plt.title("Target Distribution")
plt.xlabel("Subscribed to Term Deposit")
plt.ylabel("Count")
plt.show()

print(df["y"].value_counts(normalize=True))


## Class Balance
The target classes are imbalanced, with fewer `"yes"` outcomes than `"no"` outcomes. Because of this, accuracy alone is not sufficient. A model could achieve high accuracy simply by predicting the majority class too often. For that reason, recall, F1-score, and ROC-AUC are emphasized.


In [ ]:
df.describe(include="all").T


In [ ]:
numeric_cols_preview = df.select_dtypes(include=np.number).columns.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df["age"], bins=30, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Age Distribution")

if "campaign" in df.columns:
    sns.histplot(df["campaign"], bins=30, kde=True, ax=axes[0, 1])
    axes[0, 1].set_title("Campaign Contacts Distribution")

if "pdays" in df.columns:
    sns.histplot(df["pdays"], bins=30, kde=True, ax=axes[1, 0])
    axes[1, 0].set_title("Pdays Distribution")

if "previous" in df.columns:
    sns.histplot(df["previous"], bins=30, kde=True, ax=axes[1, 1])
    axes[1, 1].set_title("Previous Contacts Distribution")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=df, x="job", order=df["job"].value_counts(dropna=False).index, ax=axes[0])
axes[0].set_title("Job Distribution")
axes[0].tick_params(axis="x", rotation=45)

sns.countplot(data=df, x="education", order=df["education"].value_counts(dropna=False).index, ax=axes[1])
axes[1].set_title("Education Distribution")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=df, x="job", hue="y", order=df["job"].value_counts(dropna=False).index, ax=axes[0])
axes[0].set_title("Subscription by Job")
axes[0].tick_params(axis="x", rotation=45)

sns.countplot(data=df, x="marital", hue="y", order=df["marital"].value_counts(dropna=False).index, ax=axes[1])
axes[1].set_title("Subscription by Marital Status")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap")
plt.show()


## Data Preparation

In [ ]:
X = df.drop("y", axis=1)
y = df["y"].map({"no": 0, "yes": 1})

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)


## Modeling

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        y_proba = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        y_proba = None

    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    }

    return results, y_pred


In [ ]:
log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())
])

tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42, class_weight="balanced"))
])

svm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(probability=True, class_weight="balanced", random_state=42))
])


In [ ]:
results = []

for name, pipe in [
    ("Logistic Regression", log_reg_pipeline),
    ("KNN", knn_pipeline),
    ("Decision Tree", tree_pipeline),
    ("SVM", svm_pipeline)
]:
    model_result, _ = evaluate_model(name, pipe, X_train, X_test, y_train, y_test)
    results.append(model_result)

results_df = pd.DataFrame(results).sort_values(by="Recall", ascending=False)
results_df


## Cross-Validation
Cross-validation is used to reduce the risk of relying too heavily on a single train/test split.


In [ ]:
scoring_metric = "recall"

cv_scores = {
    "Logistic Regression": cross_val_score(log_reg_pipeline, X_train, y_train, cv=5, scoring=scoring_metric),
    "KNN": cross_val_score(knn_pipeline, X_train, y_train, cv=5, scoring=scoring_metric),
    "Decision Tree": cross_val_score(tree_pipeline, X_train, y_train, cv=5, scoring=scoring_metric),
    "SVM": cross_val_score(svm_pipeline, X_train, y_train, cv=5, scoring=scoring_metric)
}

cv_summary = pd.DataFrame({
    "Model": cv_scores.keys(),
    "CV Recall Mean": [scores.mean() for scores in cv_scores.values()],
    "CV Recall Std": [scores.std() for scores in cv_scores.values()]
}).sort_values(by="CV Recall Mean", ascending=False)

cv_summary


## Hyperparameter Tuning with Grid Search

In [ ]:
log_reg_param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__solver": ["liblinear", "lbfgs"]
}

log_reg_grid = GridSearchCV(
    log_reg_pipeline,
    param_grid=log_reg_param_grid,
    cv=5,
    scoring="recall",
    n_jobs=-1
)

log_reg_grid.fit(X_train, y_train)

print("Best Logistic Regression Params:", log_reg_grid.best_params_)
print("Best CV Recall:", log_reg_grid.best_score_)


In [ ]:
tree_param_grid = {
    "classifier__max_depth": [3, 5, 7, 10, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4]
}

tree_grid = GridSearchCV(
    tree_pipeline,
    param_grid=tree_param_grid,
    cv=5,
    scoring="recall",
    n_jobs=-1
)

tree_grid.fit(X_train, y_train)

print("Best Decision Tree Params:", tree_grid.best_params_)
print("Best CV Recall:", tree_grid.best_score_)


In [ ]:
knn_param_grid = {
    "classifier__n_neighbors": [3, 5, 7, 9],
    "classifier__weights": ["uniform", "distance"],
    "classifier__metric": ["minkowski"]
}

knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid=knn_param_grid,
    cv=5,
    scoring="recall",
    n_jobs=-1
)

knn_grid.fit(X_train, y_train)

print("Best KNN Params:", knn_grid.best_params_)
print("Best CV Recall:", knn_grid.best_score_)


In [ ]:
svm_param_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__kernel": ["linear", "rbf"]
}

svm_grid = GridSearchCV(
    svm_pipeline,
    param_grid=svm_param_grid,
    cv=3,
    scoring="recall",
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

print("Best SVM Params:", svm_grid.best_params_)
print("Best CV Recall:", svm_grid.best_score_)


## Final Tuned Model Comparison

In [ ]:
best_models = {
    "Logistic Regression": log_reg_grid.best_estimator_,
    "KNN": knn_grid.best_estimator_,
    "Decision Tree": tree_grid.best_estimator_,
    "SVM": svm_grid.best_estimator_
}

final_results = []

for name, model in best_models.items():
    result, _ = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    final_results.append(result)

final_results_df = pd.DataFrame(final_results).sort_values(by="Recall", ascending=False)
final_results_df


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=final_results_df, x="Model", y="Recall")
plt.title("Model Comparison by Recall")
plt.ylabel("Recall")
plt.xlabel("Model")
plt.ylim(0, 1)
plt.show()


In [ ]:
best_model_name = final_results_df.iloc[0]["Model"]
best_model = best_models[best_model_name]

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Best Model:", best_model_name)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix: {best_model_name}")
plt.show()


## Model Interpretation
Logistic Regression offers the clearest coefficient-based interpretation, so coefficient inspection is included below.


In [ ]:
best_log_reg = log_reg_grid.best_estimator_
best_log_reg.fit(X_train, y_train)

ohe = best_log_reg.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
encoded_cat_names = ohe.get_feature_names_out(categorical_cols)
all_feature_names = np.concatenate([numeric_cols, encoded_cat_names])

coefficients = best_log_reg.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", ascending=False)

coef_df.head(15)


In [ ]:
top_positive = coef_df.head(10)
top_negative = coef_df.tail(10)
coef_plot_df = pd.concat([top_positive, top_negative])

plt.figure(figsize=(10, 8))
sns.barplot(data=coef_plot_df, x="Coefficient", y="Feature")
plt.title("Top Positive and Negative Logistic Regression Coefficients")
plt.show()


# Findings

## Business Understanding
The bank wants to improve the efficiency of direct marketing campaigns by identifying customers who are more likely to subscribe to a term deposit. This can reduce wasted effort and improve campaign effectiveness.

## Modeling Summary
Four classifiers were compared:
- Logistic Regression
- K-Nearest Neighbors
- Decision Tree
- Support Vector Machine

Each model was evaluated using recall, F1-score, ROC-AUC, and accuracy, with emphasis on recall because identifying likely subscribers is more valuable than simply maximizing overall accuracy.

## Key Findings
- Logistic Regression provided strong interpretability and competitive performance.
- Decision Trees were easy to explain but showed greater risk of overfitting.
- KNN was less practical due to computational cost and sensitivity to scaling.
- SVM performed well but required more computational effort and was harder to interpret.

## Actionable Insights
The bank can use the best-performing model to:
- Prioritize customers who are more likely to subscribe
- Reduce calls to customers with low likelihood of conversion
- Improve campaign efficiency and reduce operational costs

## Important Variables
Based on model interpretation, factors such as previous campaign outcome, contact timing, and customer profile can influence subscription likelihood.

## Recommendations
- Deploy the best-performing model to support campaign targeting
- Continue tuning model thresholds to balance recall and precision
- Explore ensemble methods such as Random Forest or Gradient Boosting in future work
- Compare results with and without different feature sets


# Next Steps

1. Test ensemble models such as Random Forest, XGBoost, or Gradient Boosting.
2. Explore resampling methods to better address class imbalance.
3. Perform threshold tuning to align predictions with business priorities.
4. Build a simple dashboard for business users to review customer subscription probabilities.
5. Validate the model on newer campaign data to confirm performance over time.


# Conclusion

This analysis compared KNN, Logistic Regression, Decision Trees, and SVM on the bank marketing dataset. Because the business goal is to identify likely subscribers, recall was used as a key evaluation metric. The final results should be interpreted by balancing model performance with interpretability and business usability.

From a business perspective, this project provides a practical way to make marketing campaigns more efficient, reduce unnecessary customer contact, and improve subscription conversion rates.
